# A1.17 · Attacks that target the humans

**Function A — Securing AI Architectures → CyberTravels' Architecture, and Every Risk It Carries**  ·  *Security of AI*

Builds on **[A1.16 · Misaligned and deceptive behaviour](https://spbreed.github.io/cyber-commons/lessons/A1.16.html)**.

| | |
|---|---|
| Tools used | standard library only |

## What this lesson is

**What it covers.** Launder a request through a delegation chain to reach something the requester was denied.

**Why a security engineer needs it.** The delegation chain is used as a privilege-laundering path, and the agent's output becomes an unusually persuasive channel into a human decision. The control it builds is: ceiling-bound delegation (A2.3), attribution per hop (A2.7) and marking machine-generated output as such (A3.6).

This is a **risk** lesson: it shows the failure happening before anything tries to stop it, so the control that follows is answering something you have already watched go wrong.

## 1 · The hook

Two of the fifteen risks in this chapter route through people rather than components: an insider using an agent to reach what they could not reach directly, and an agent whose output is persuasive enough to move a human decision. No control in chapter 3 touches either.

> **At CyberTravels.** An employee who cannot issue refunds asks CyberTravels to, and it can. And a confident itinerary from the advisor moves an executive's decision without anyone checking it. Neither is closed by anything in chapter 3.

## 2 · The framework

```
   through people, not components

   insider --> agent --> resource the insider could not reach directly
                            (the agent's authority, not theirs)

   agent --> confident output --> human --> decision
                            (persuasion, not compromise)

   no control in chapter 3 touches either of these
```

**OWASP T14 — Human Attacks on Multi-Agent Systems. T15 — Human Manipulation.**

The last two threats route through people rather than components, and they run
in opposite directions.

**T14 — a human attacking the system.** An insider does not need to defeat
authorization. They need to find a delegation path where authority is composed.
Ask the orchestrator for something it will route to an agent that holds a
credential you do not. Each hop is individually legitimate — you were allowed to
ask, the orchestrator was allowed to route, the agent was allowed to act — and
the composition reaches something you were explicitly denied. Privilege
laundering, using the architecture exactly as designed.

**T15 — the system manipulating a human.** The output of an agent arrives with
institutional authority. It is formatted like a report, it cites things, it does
not hedge. A person reading it makes a decision on it, and applies less scrutiny
than they would to a colleague's opinion — because it looks like a system
output rather than an argument.

That is exploitable in both directions: an attacker who lands an injection at
A1.3 gets their content delivered in your agent's trusted voice, and an agent
that is merely wrong at A1.12 gets the same credibility for free.

The uncomfortable version: the more your agent is trusted, the more valuable it
becomes as a channel into human decisions — so success at deployment increases
this risk rather than reducing it.

> **Where this lands on the reference architecture.**
>
> ```
> ingress -> orchestrator -> agent_runtime -> model
>                                |              |
>                          messaging        tools / mcp
>                                |              |
>                       knowledge / memory   egress
>            identity + policy wrap every call · observability records it
> ```

## 3 · The risk, realised

A request that is denied directly, and permitted through the architecture.

## 4 · The check, as a skill

A CyberTravels traveller denied `payments:write` reaches it through the orchestrator, with every hop legitimate. The skill records the direct refusal first, so the composed success reads as a finding rather than as intended behaviour.

### The skill — [`skills/threats/authority-composition-check/SKILL.md`](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/skills/threats/authority-composition-check/SKILL.md)

```yaml
name: authority-composition-check
description: >-
  Check whether a scope a user is denied directly can be reached through a chain
  of individually legitimate hops, and whether an agent's restatement of a claim
  carries more weight than a colleague's. Use when reviewing an orchestrator, a
  routing layer, or an agent that answers on behalf of people.
allowed-tools: Read, Grep, Glob
```

# Every hop legitimate, the composition unauthorised

A user is denied `payments:write` at the front door and reaches it through the
orchestrator. No hop broke a rule. Authorisation was decided per edge, and
nobody owns the path. The second half of the check is social: the same claim is
believed more when an agent states it, which is what makes an agent a good
carrier for one.

## When to use this

Any system where a request is routed, decomposed, or delegated between
components with different privileges — an orchestrator, a workflow engine, a
tool that calls another tool.

## Procedure

**1 — Draw the edges with their scopes.** For each hop, what identity it runs
as and what it may do. The picture is usually the first time anyone has seen
the composition.

**2 — Attempt the direct call and record the refusal.** Establish that the
control exists, so the composed success is a finding rather than a
misunderstanding.

**3 — Compose a path to the same effect.** Ask for something the orchestrator
will decompose into the denied action. Record each hop's decision: every one
should be a legitimate "yes".

**4 — Find where the path could have been evaluated.** Usually nowhere: each
component sees one edge. Name the component that would have to hold the
end-to-end policy, because that is the fix.

**5 — Test the trust asymmetry.** Present the same unverified claim from an
agent and from a person, and record which is challenged. If the agent's version
is accepted more readily, the agent is the better delivery vehicle and that
belongs in the report.

## Example

**Input** — the fixture committed at the top of [`scripts/authority_composition_check.py`](scripts/authority_composition_check.py). Edit it and re-run: the buckets, counts and verdicts below are derived from it, not hard-coded.

**Output** — the opening lines of a real run:

```
mallory holds        : ['reports:read']
mallory asks directly for payments:write -> DENIED

same outcome, requested through the architecture:
   user asks orchestrator    mallory         ok
   orchestrator routes       orchestrator    ok
   agent acts                finance-agent   ok
   -> reached payments:write: True
```

The run continues past this. The script is the example: `test_skills.py` executes it on every build, so this block cannot drift from what the skill actually prints.

## Output contract

```json
{
  "edges": [{"from": "str", "to": "str", "runs_as": "str", "scopes": ["str"]}],
  "direct": {"scope": "str", "refused": true},
  "composed": {"path": ["str"], "each_hop_legitimate": true, "effect_achieved": true},
  "policy_owner": {"component": "str", "exists": false},
  "trust_asymmetry": {"agent_claim_challenged": false, "human_claim_challenged": true}
}
```

## Failure modes

- **Auditing hops.** They are all fine; that is the point.
- **Skipping the direct refusal.** Without it the composed success looks like
  intended behaviour.
- **Leaving the trust asymmetry out** because it is not technical. It is the
  reason the finding recurs.

In [ ]:
# The code is not in this notebook. It is this file in the repository:
#   https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/skills/threats/authority-composition-check/scripts/authority_composition_check.py
SCRIPT = "skills/threats/authority-composition-check/scripts/authority_composition_check.py"
REPO = "https://github.com/spbreed/cyber-commons"
BRANCH = "claude/vulnbench-setup-scheduling-81aqov"

import glob, os, subprocess, sys

CLONE = "/kaggle/working/cyber-commons"
_root = next((r for r in (".", "..", "../..", CLONE)
              if os.path.isfile(os.path.join(r, SCRIPT))), None)

if _root is None:
    # --filter=blob:none --sparse fetches the tree without the history or the
    # notebooks; `sparse-checkout set skills` then materialises only what runs.
    _c = subprocess.run(["git", "clone", "--depth", "1", "--filter=blob:none",
                         "--sparse", "--branch", BRANCH, REPO, CLONE],
                        capture_output=True, text=True)
    if _c.returncode:
        raise SystemExit(
            "could not fetch the skills: " + _c.stderr.strip()[-300:] +
            "\nOn Kaggle this needs Internet on in the notebook settings, which "
            "needs a phone-verified account. Without one, attach the dataset "
            "cybercommons/cyber-commons-skills instead — it holds the same tree.")
    subprocess.run(["git", "-C", CLONE, "sparse-checkout", "set", "skills"],
                   capture_output=True, text=True)
    _root = CLONE

_out = subprocess.run([sys.executable, os.path.join(_root, SCRIPT)],
                      capture_output=True, text=True,
                      env=dict(os.environ,
                               PYTHONPATH=os.path.join(_root, "skills/_runtime"),
                               PYTHONHASHSEED="0"))
print(_out.stdout, end="")
if _out.returncode:
    raise SystemExit(_out.stderr.strip()[-2000:])

## What you just proved

A user denied `payments:write` directly reaches it through the orchestrator, with every individual hop legitimate and only the composition unauthorised — and the same claim is shown carrying more weight when an agent states it than when a colleague does.

## Your turn

Take one permission a user is denied and see whether an agent they can talk to holds it. That pair is a laundering path, and it is invisible to any review that checks permissions one hop at a time.

---

**Next → [A1.18 · The CyberTravels risk register](https://spbreed.github.io/cyber-commons/lessons/A1.18.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/A1.17.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/A1.17.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*